<a href="https://colab.research.google.com/github/Satyadeep-Dey/genai-lab/blob/main/3_OpenAI_%2B_Chat_%2B_Gradio_UI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install necessary libraries
!pip install openai gradio --quiet

In [ ]:
# Import required libraries

import requests
import json
from openai import OpenAI
from google.colab import userdata
import gradio as gr

#define constants
MODEL = "gpt-4.1-mini"
#MODEL = "gpt-5.2"

In [ ]:
# Sign in to OpenAI using Secrets in Colab

openai_api_key = userdata.get('OPENAI_API_KEY')
openai = OpenAI(api_key=openai_api_key)

In [ ]:
# Check if Open AI key has been set
if openai_api_key:
    print("OpenAI API Key exists")
else:
    print("OpenAI API Key not set")

In [ ]:
system_message = "You are a chatbot . Keep you answers short. If you don't know the answer then respond with DONT-KNOW"

In [ ]:
history = []
# In the Chat Completions API, the history field must be a list of message dictionaries.


In [ ]:
# the function needs to have this name and signature
def chatter(message, history):

    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]

    # history is added to messages so that context is passed between questions in chat
    # Example : Q1 - who is Sachin Tendulkar ?
    #           Q2 - what's his age ?

    completion = openai.chat.completions.create(
        model=MODEL,
        messages=messages
       )

    reply = completion.choices[0].message.content

    # Update history
    history.append({"role": "user", "content": message})
    history.append({"role": "assistant", "content": reply})

    return reply


In [ ]:
history = []

print(chatter("What is the capital of Goa", history))
print(chatter("What's my name?", history))
print(chatter("What's the name of the Hero in Tale of Two Cities ?", history))
print("********************************************************************")
print(chatter("Who is Sachin Tendulkar ?",history))
print(chatter("what's his age ?",history)) # LLM knows the context because we're passing history and so can answer
print("********************************************************************")
print("History of chat is as follows :")

for history in history:
  print(history)


In [ ]:
# function should not append history when using Gradio since Gradio takes care of it
# but it should have a paremeter called history which should be added to message for context
def gradio_chatter(message,history):

    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    # history is added to messages so that context is passed between questions in chat
    # Example : Q1 - who is Sachin Tendulkar ?
    #           Q2 - what's his age ?
    completion = openai.chat.completions.create(
        model=MODEL,
        messages=messages
       )

    return completion.choices[0].message.content


In [ ]:
# Let's use Gradio to make a chat interface

chat = gr.ChatInterface(
    fn=gradio_chatter,
    type="messages",
    title="🤖 AI Assistant",
    description="Your intelligent assistant !!",
    theme=gr.themes.Default(
        primary_hue="indigo",
        secondary_hue="cyan",
        neutral_hue="slate",
    ),
    css="""
    textarea {
        border: 2px solid #4f46e5 !important;
        border-radius: 8px !important;
    }
    textarea:focus {
        border: 2px solid #06b6d4 !important;
        box-shadow: 0 0 6px #06b6d4 !important;
    }
    """
)

chat.launch(pwa=True, inbrowser=True, share=True)


In [ ]:
chat.close()